# Basic comparsion of naive and fast convolution

In [2]:
import numpy as np
import sympy as sy
from scipy import signal
from scipy import datasets
from matplotlib import pyplot as plt

In [3]:
from naive_convolve import naive_convolve
from fast_convolution import c3x3_5m20a9e
from utils import plot_pdf, symmetrical_cyclic_convolution

Lets do a really basic test to compare naive with fast convolution
The input feature will be a simple 5x5 matrix and the output will be a 3x3 matrix

In [4]:
feature = np.arange(8*8).reshape(8, 8)
feature

array([[ 0,  1,  2,  3,  4,  5,  6,  7],
       [ 8,  9, 10, 11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20, 21, 22, 23],
       [24, 25, 26, 27, 28, 29, 30, 31],
       [32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47],
       [48, 49, 50, 51, 52, 53, 54, 55],
       [56, 57, 58, 59, 60, 61, 62, 63]])

In [5]:
weight = np.array([
    [ 1, 0, 1],
    [ 2, 1, 2],
    [ 1, 2, 1],
])
weight

array([[1, 0, 1],
       [2, 1, 2],
       [1, 2, 1]])

In [6]:
np.fliplr(weight)

array([[1, 0, 1],
       [2, 1, 2],
       [1, 2, 1]])

Convolution with no paddin and stride 1
Using convolve2d from scipy, our gold method, it's necessary to reverse the weights to get same result of naive and fast convolution

In [7]:
weight_reversed = weight[::-1, ::-1]
weight_reversed

array([[1, 2, 1],
       [2, 1, 2],
       [1, 0, 1]])

In [8]:
output = signal.convolve2d(feature, weight_reversed, mode='valid')
output

array([[115, 126, 137, 148, 159, 170],
       [203, 214, 225, 236, 247, 258],
       [291, 302, 313, 324, 335, 346],
       [379, 390, 401, 412, 423, 434],
       [467, 478, 489, 500, 511, 522],
       [555, 566, 577, 588, 599, 610]])

Running naive convolution
9 multiplications and 8 aditions per output scalar

In [56]:
output_naive = naive_convolve(feature, weight)
output_naive

array([[115, 126, 137, 148, 159, 170],
       [203, 214, 225, 236, 247, 258],
       [291, 302, 313, 324, 335, 346],
       [379, 390, 401, 412, 423, 434],
       [467, 478, 489, 500, 511, 522],
       [555, 566, 577, 588, 599, 610]])

In [57]:
np.all(output == output_naive)

True

Run one fast convolution for each 1d kernel

In [9]:
feature[0]

array([0, 1, 2, 3, 4, 5, 6, 7])

In [109]:
size = 5
step = 3

In [25]:
np.pad(feature[0][r: r+size], (0, size - len(feature[0][r: r+size])), 'constant', constant_values=0)

array([5, 6, 7, 0, 0])

In [27]:
np.pad([0,0,0,0,0], (0, 0), 'constant', constant_values=0)

array([0, 0, 0, 0, 0])

In [86]:
feature[0][0]

0

How join multiple 1d convolution in one 2d convolution

In [91]:
c0 = np.convolve(feature[0], weight[0])
c0

array([ 0,  1,  2,  4,  6,  8, 10, 12,  6,  7])

In [92]:
c1 = np.convolve(feature[1], weight[1])
c1

array([16, 26, 45, 50, 55, 60, 65, 70, 43, 30])

In [93]:
c2 = np.convolve(feature[2], weight[2])
c2

array([16, 49, 68, 72, 76, 80, 84, 88, 68, 23])

In [94]:
c0+c1+c2

array([ 32,  76, 115, 126, 137, 148, 159, 170, 117,  60])

In [110]:
fast_conv0 = c3x3_5m20a9e(weight[0].tolist())
out0 = []
r=0
for c in range(0, feature.shape[0], step):
    fast_conv0(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()

In [108]:
feature[0][r: r+size] 

array([0, 1, 2])

In [113]:
fast_conv0 = c3x3_5m20a9e(weight[0].tolist())
out0 = [
    # np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)
    fast_conv0(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()
    for r in range(0, feature.shape[1]-2)
    for c in range(0, feature.shape[0]-step, step)
]
out0

[[2, 4, 6],
 [8, 10, 12],
 [18, 20, 22],
 [24, 26, 28],
 [34, 36, 38],
 [40, 42, 44],
 [50, 52, 54],
 [56, 58, 60],
 [66, 68, 70],
 [72, 74, 76],
 [82, 84, 86],
 [88, 90, 92]]

In [114]:
fast_conv1 = c3x3_5m20a9e(weight[1].tolist())
out1 = [
    # feature[r][c: c+size]
    fast_conv1(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()
    for r in range(1, feature.shape[1]-1)
    for c in range(0, feature.shape[0]-step, step)
]
out1

[[45, 50, 55],
 [60, 65, 70],
 [85, 90, 95],
 [100, 105, 110],
 [125, 130, 135],
 [140, 145, 150],
 [165, 170, 175],
 [180, 185, 190],
 [205, 210, 215],
 [220, 225, 230],
 [245, 250, 255],
 [260, 265, 270]]

In [115]:
fast_conv2 = c3x3_5m20a9e(weight[2].tolist())
out2 = [
    fast_conv2(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()
    for r in range(2, feature.shape[1])
    for c in range(0, feature.shape[0]-step, step)
]
out2

[[68, 72, 76],
 [80, 84, 88],
 [100, 104, 108],
 [112, 116, 120],
 [132, 136, 140],
 [144, 148, 152],
 [164, 168, 172],
 [176, 180, 184],
 [196, 200, 204],
 [208, 212, 216],
 [228, 232, 236],
 [240, 244, 248]]

Sum results in the first dimension

In [117]:
output_fast = np.sum([out0, out1, out2], axis=0).reshape(output.shape)
output_fast

array([[115, 126, 137, 148, 159, 170],
       [203, 214, 225, 236, 247, 258],
       [291, 302, 313, 324, 335, 346],
       [379, 390, 401, 412, 423, 434],
       [467, 478, 489, 500, 511, 522],
       [555, 566, 577, 588, 599, 610]], dtype=object)

In [118]:
np.all(output_fast == output_naive)

True

Camparing how much operations are used in naive and fast method

Output Size

In [119]:
size = output.size
size

36

Naive: total of multiplications

In [121]:
size * 9

324

Naive: total of additions

In [122]:
size * 8

288

Fast: total of multiplications

In [123]:
size * 5

180

Fast: additions for each batch processed

In [124]:
add0 = size * 20
add0

720

Fast: additions to join batches

In [125]:
add1 = size * 2
add1

72

Fast: Total of additions

In [126]:
add0 + add1

792

Fast: total of extra operations - bit shifts and etc

In [127]:
size * 9

324